In [16]:
import os
import base64
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_core.embeddings import Embeddings
import pymupdf
from dotenv import load_dotenv
import cohere

load_dotenv()

True

### PDF TO Images

In [2]:
def pdf_to_images(filepath):
    if not filepath:
        return "File path is not Given"

    docs = pymupdf.open(filepath)
    pages = len(docs)

    urls ,pngs = [],[]

    for i in range(pages):
        page = docs[i]

        png = page.get_pixmap().tobytes('png')

        b64 = base64.b64encode(png).decode()

        url = f"data:image/png;base64,{b64}"

        pngs.append(png)
        urls.append(url)

    return urls, pngs

In [3]:
page_url, page_png = pdf_to_images(rf"C:\Users\Edin\Downloads\Aditya_Report.pdf")


### Cohere Image Embedding

In [4]:
class Cohere(Embeddings):
    def __init__(self,model = "embed-v4.0"):
        self.client = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))
        self.model = model

    def embed_documents(self, images):
        if not images:
            return []
        result = self.client.embed(
            model = self.model,
            images = images,
            input_type = "search_document",
            embedding_types=['float']
        )
        vectors = [list(v) for v in result.embeddings.float_]
        return vectors

    def embed_query(self,text):
          if not text:
                    return []
          result = self.client.embed(
                    model = self.model,
                    texts=[text],
                    input_type = "search_query",
                    embedding_types=['float']
                )
          vectors = [list(v) for v in result.embeddings.float_][0]
          return vectors


### Embed all Page

In [ ]:
embeding = Cohere()
page_vector = embeding.embed_documents(page_url)

len(page_vector[0])

39

### Qdrant

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PointStruct

c:\Users\Edin\Desktop\MultiModal_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
client = QdrantClient(url="http://localhost:6333")
DIM = len(page_vector[0])
COLLECTIONNAME = "page_query"
if  not client.collection_exists(COLLECTIONNAME):
    client.create_collection(
        collection_name=COLLECTIONNAME,
        vectors_config=VectorParams(size=DIM,distance= Distance.COSINE)
    )



In [8]:
points = []
for i,vector in enumerate(page_vector):
    point = PointStruct(
        id = i,
        vector = vector,
        payload={"page_no." : i}
    )
    points.append(point)


client.upsert(collection_name=COLLECTIONNAME,points=points)


UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [39]:
query = "Explain  System Flow Diagram ??"
q_vector = embeding.embed_query(query)


In [40]:
records = client.query_points(collection_name=COLLECTIONNAME,query=q_vector,limit=5).points
top_pages = []
for record in records:
    pagenum = record.payload.get("page_no.")
    top_pages.append(pagenum)


top_pages

[22, 7, 19, 32, 31]

In [41]:
img = page_url[top_pages[0]]


In [42]:
llm = ChatGroq(model="qwen/qwen3.8-27b",api_key=os.getenv("GROQ_API_KEY"))


In [43]:
prompt = [
    {
        'role' : 'user',
        "content" : [
             {
                 "type" : "text","text" : f"Answer only from Image Context . Question : {query}"
             },
             {
                 "type" : "image_url","image_url" : {"url" : img}
             }
        ]
    }
]

In [44]:
res = llm.invoke(prompt)


In [45]:
res.content

'Based **only** on the provided image context (specifically Section 4.1 and Figure 4.1), here is an explanation of the System Flow Diagram:\n\nThe System Flow Diagram (Figure 4.1: System Flow) illustrates the **AI Application Data Flow & System Architecture** for the **Solar Yojna AI system**, which uses a **Retrieval-Augmented Generation (RAG)** architecture.\n\nThe flow proceeds as follows:\n\n1.  **USER INTERACTION**: The user initiates the process.\n2.  **1. USER INPUT**: The user\'s input/query is sent from the user interaction point.\n3.  **WEB UI (HTML, CSS, JS)**: The input is received by the web-based interface, developed using HTML, CSS, and JavaScript.\n4.  **2. REQUEST**: The web UI sends the user\'s query as a request to the backend.\n5.  **FastAPI API**: The request is received and processed by the FastAPI backend.\n6.  **3a. QUERY**: The FastAPI backend sends the user\'s query to the ChromaDB vector database.\n7.  **ChromaDB VECTOR DATABASE / STORAGE**: This component st